In [1]:
#install llama-cpp-python on colab w gpu support, optional for gguf models
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

  Using cached llama_cpp_python-0.3.16.tar.gz (50.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
Failed to build llama-cpp-python
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (llama-cpp-python)


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os

# Google AI Studio imports
try:
    from google import genai
    from google.colab import userdata
    print("Google AI Studio libraries imported.")
except ImportError:
    genai = None
    print("Google AI Studio libraries not found. Run `pip install -U google-generativeai` if needed.")

# google drive import
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ReadmeGenerator/

# Detect device
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Try to import llama_cpp for GGUF support
try:
    from llama_cpp import Llama
    print("llama-cpp-python is available.")
except ImportError:
    Llama = None
    print("llama-cpp-python is NOT installed. GGUF models will not be supported.")

Google AI Studio libraries imported.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/ReadmeGenerator
Using device: cpu
llama-cpp-python is NOT installed. GGUF models will not be supported.


Declaring the Quen3-Coder-Next Model and Loading the Tokenizer

In [3]:
model_name = "aistudio-gemini-2.5-flash"
# Example GGUF usage: model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
# Example Gemini usage: model_name = "aistudio-gemini-1.5-flash"

tokenizer = None
model = None

if "gguf" in model_name.lower():
    if Llama is None:
        raise ImportError("You requested a GGUF model but llama-cpp-python is not installed. Please install it with `pip install llama-cpp-python`.")

    print(f"Loading GGUF model: {model_name} using llama-cpp-python...")
    # Attempt to load local path or download from Hub
    if os.path.exists(model_name):
        model = Llama(model_path=model_name, n_gpu_layers=-1, verbose=False)
    else:
        # Downloads a generic quantized version if not specified
        model = Llama.from_pretrained(
            repo_id=model_name,
            filename="*Q4_K_M.gguf",
            verbose=False,
            n_gpu_layers=-1 # Offload to GPU if available
        )
    print("GGUF Model loaded successfully!")

elif model_name.startswith("aistudio-"):
    if genai is None:
         raise ImportError("Google Generative AI SDK not installed. Please install it.")

    real_model_name = model_name.replace("aistudio-", "")
    print(f"Configuring Google AI Studio model: {real_model_name}...")

    try:
        GOOGLE_API_KEY = userdata.get('google_aistudio_key')
        client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Google AI Studio Model configured successfully!")
        try:
            for m in client.models.list():
                print(m.name)
        except Exception as e:
            print(f"Error listing models: {e}")
    except Exception as e:
        print(f"Error configuring Google AI Studio: {e}. Make sure 'google_aistudio_key' is set in Colab secrets.")
        model = None

else:
    # Standard Transformers loading
    print(f"Loading Transformers model: {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    print("Transformers Model loaded successfully!")

Configuring Google AI Studio model: gemini-2.5-flash...
Google AI Studio Model configured successfully!
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-p

README Generation

In [32]:
def generate_readme(parsed_file, model, tokenizer):
    # Check if the parsed file exists
    if not os.path.exists(parsed_file):
        print(f"File {parsed_file} does not exist.")
        return None

    with open(parsed_file, 'r', encoding='utf-8', errors='ignore') as f:
        code_content = f.read()

    # Create a prompt
    prompt = f"Based on the following repository code structure and contents, generate a comprehensive README.md file. Include sections for: Description, Features, Installation, Usage, and any other relevant information.\n\n{code_content}\n\n"

    readme_content = ""

    # --- Inference Logic ---

    # 1. Google AI Studio (Gemini)
    # We check if 'client' is available in globals, which indicates the aistudio setup was run.
    if 'client' in globals() and client is not None and (model is None or hasattr(model, 'generate_content')):
        try:
            print(f"Generating content with Google AI Studio model: {real_model_name}...")
            # Using the updated client.models.generate_content syntax
            response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
            readme_content = response.text
        except Exception as e:
            print(f"Error generating content with Gemini: {e}")
            return ""

    # 2. llama-cpp-python
    elif model is not None and hasattr(model, 'create_completion'):
        print("Generating content with llama-cpp...")
        output = model.create_completion(
            prompt,
            max_tokens=4096,
            temperature=0.7,
            top_p=0.9,
            echo=False
        )
        readme_content = output['choices'][0]['text']

    # 3. Hugging Face Transformers
    elif model is not None and tokenizer is not None:
        print("Generating content with Transformers...")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=16384  # Adjusted for potentially large context
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4096,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        readme_content = tokenizer.decode(outputs[0], skip_special_tokens=True)

    else:
        print("Error: No valid model configuration or client detected.")
        return ""

    # Post-processing
    if "README.md" in readme_content:
        # Attempt to split to get content after the header if the model echoes it
        parts = readme_content.split("README.md", 1)
        if len(parts) > 1:
            readme_content = parts[1].strip()

    # Clean up code blocks if the model wrapped the output
    if readme_content.startswith("```markdown"):
        readme_content = readme_content.replace("```markdown", "", 1)
    elif readme_content.startswith("```"):
        readme_content = readme_content.replace("```", "", 1)

    if readme_content.endswith("```"):
        readme_content = readme_content[:-3]

    return readme_content.strip()

Saving the README in llm_output directory

In [33]:
def save_readme(output_path, readme_content, parsed_file, prefix=""):
    # Saves the generated README.md content to a file in the specified output path
    original_filename = os.path.basename(parsed_file)
    base_name = os.path.splitext(original_filename)[0]
    filename = f"{prefix}{base_name}_README.md" if prefix else f"{base_name}_README.md"
    output_file = os.path.join(output_path, filename)

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_file}")
    return output_file

Generating the README from a repository

In [34]:
parsed_file = "./out/apache_cordova-plugin-splashscreen_parsed_code.txt"
output_path = "llm_output"

readme = generate_readme(parsed_file, model, tokenizer)
print(readme)


Generating content with Google AI Studio model: gemini-2.5-flash...
` is generated based on the provided repository code structure and contents.

---

# cordova-plugin-splashscreen

[![Chrome Testsuite](https://github.com/apache/cordova-plugin-splashscreen/actions/workflows/chrome.yml/badge.svg)](https://github.com/apache/cordova-plugin-splashscreen/actions/workflows/chrome.yml)
[![Lint Test](https://github.com/apache/cordova-plugin-splashscreen/actions/workflows/lint.yml/badge.svg)](https://github.com/apache/cordova-plugin-splashscreen/actions/workflows/lint.yml)

The `cordova-plugin-splashscreen` is an Apache Cordova plugin that enables the display and control of a splash screen during your web application's launch process. It provides methods to show and hide the splash screen manually, and offers configuration options to customize its appearance and behavior.

**Note**: As of version 6.0.0, this repository primarily provides the splash screen implementation for the **Browser** plat

In [35]:
save_readme(output_path, readme, parsed_file)

README.md has been saved to llm_output/apache_cordova-plugin-splashscreen_parsed_code_README.md


'llm_output/apache_cordova-plugin-splashscreen_parsed_code_README.md'